In [0]:
# ============================================================
# Silver — Source 02: Debezium CDC
#
# Captures every change to Postgres as an audit trail.
# Tables captured: orders, order_items, payments
# op=r (snapshot) filtered out — already in Silver 01
# op=c (insert), op=u (update), op=d (delete) kept
#
# Routes by cdc_table to extract correct fields per table.
# Dedup rule: if order_id exists in both Silver 01 and Silver 02
#             → CDC version wins (more recent)
#
# Source:  bronze.src_02_cdc.events
# Targets: silver.src_02_cdc.order_changes
#          silver.src_02_cdc.order_item_changes
#          silver.src_02_cdc.payment_changes
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import re as re_lib

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_02_cdc')
print('Silver Source 02 CDC — starting...')


In [0]:
# ── LOAD AND FILTER BRONZE CDC ────────────────────────────────
cdc_bronze = spark.table(f'{BRONZE_CATALOG}.src_02_cdc.events')
total = cdc_bronze.count()
print(f'Bronze CDC total rows: {total}')
cdc_bronze.groupBy('cdc_op', 'cdc_table').count().orderBy('cdc_table', 'cdc_op').show()

# Filter — only real changes, not snapshots
changes = cdc_bronze.filter(F.col('cdc_op').isin(['c', 'u', 'd']))
change_count = changes.count()
print(f'Live change events (op=c/u/d): {change_count}')

if change_count == 0:
    print('\nNo live CDC changes yet — MSK not running.')
    print('All rows are op=r (snapshot) — already captured in Silver 01.')
    print('This notebook will produce rows when MSK is recreated.')
    dbutils.notebook.exit('No live CDC changes — skipping Silver write')


In [0]:
# ── REGEX HELPERS ─────────────────────────────────────────────
from pyspark.sql.functions import udf

@udf(returnType=LongType())
def ext_long(s, field):
    if not s: return None
    try:
        m = re_lib.search(rf'(?<![\w]){field}=([-\d]+)', s)
        return int(m.group(1)) if m else None
    except: return None

@udf(returnType=StringType())
def ext_str(s, field):
    if not s: return None
    try:
        m = re_lib.search(rf'(?<![\w]){field}=([^,}}]+)', s)
        v = m.group(1).strip() if m else None
        return None if v in ('null', 'NULL', '') else v
    except: return None

print('UDFs registered')


In [0]:
# ── ROUTE BY TABLE AND PARSE ──────────────────────────────────

# ── ORDERS ────────────────────────────────────────────────────
order_changes = changes.filter(F.col('cdc_table') == 'orders') \
    .withColumn('order_id',     ext_long(F.col('after_raw'), F.lit('order_id'))) \
    .withColumn('customer_id',  ext_long(F.col('after_raw'), F.lit('customer_id'))) \
    .withColumn('order_status', ext_str(F.col('after_raw'),  F.lit('order_status'))) \
    .withColumn('subtotal_pence', ext_long(F.col('after_raw'), F.lit('subtotal_pence'))) \
    .withColumn('discount_pence', ext_long(F.col('after_raw'), F.lit('discount_pence'))) \
    .withColumn('tax_pence',     ext_long(F.col('after_raw'), F.lit('tax_pence'))) \
    .withColumn('shipping_amount_pence', ext_long(F.col('after_raw'), F.lit('shipping_amount_pence'))) \
    .withColumn('channel',      ext_str(F.col('after_raw'),  F.lit('channel'))) \
    .withColumn('changed_at',   (F.col('cdc_ts_ms') / 1000).cast('timestamp')) \
    .select('cdc_event_id', 'cdc_op', 'changed_at', 'order_id', 'customer_id',
            'order_status', 'subtotal_pence', 'discount_pence', 'tax_pence',
            'shipping_amount_pence', 'channel', 'after_raw')

order_count = order_changes.count()
print(f'Order changes: {order_count}')

if order_count > 0:
    TARGET = f'{SILVER_CATALOG}.src_02_cdc.order_changes'
    if spark.catalog.tableExists(TARGET):
        dt = DeltaTable.forName(spark, TARGET)
        dt.alias('t').merge(order_changes.alias('s'), 't.cdc_event_id = s.cdc_event_id') \
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        order_changes.write.format('delta').mode('overwrite').saveAsTable(TARGET)
    print(f'✅ {TARGET}: {order_count} rows')

# ── ORDER ITEMS ───────────────────────────────────────────────
item_changes = changes.filter(F.col('cdc_table') == 'order_items') \
    .withColumn('item_id',          ext_long(F.col('after_raw'), F.lit('item_id'))) \
    .withColumn('order_id',         ext_long(F.col('after_raw'), F.lit('order_id'))) \
    .withColumn('product_sku',      ext_str(F.col('after_raw'),  F.lit('product_sku'))) \
    .withColumn('quantity',         ext_long(F.col('after_raw'), F.lit('quantity'))) \
    .withColumn('unit_price_pence', ext_long(F.col('after_raw'), F.lit('unit_price_pence'))) \
    .withColumn('total_price_pence',ext_long(F.col('after_raw'), F.lit('total_price_pence'))) \
    .withColumn('is_refunded',      ext_str(F.col('after_raw'),  F.lit('is_refunded'))) \
    .withColumn('changed_at',       (F.col('cdc_ts_ms') / 1000).cast('timestamp')) \
    .select('cdc_event_id', 'cdc_op', 'changed_at', 'item_id', 'order_id',
            'product_sku', 'quantity', 'unit_price_pence', 'total_price_pence',
            'is_refunded', 'after_raw')

item_count = item_changes.count()
print(f'Order item changes: {item_count}')

if item_count > 0:
    TARGET = f'{SILVER_CATALOG}.src_02_cdc.order_item_changes'
    if spark.catalog.tableExists(TARGET):
        dt = DeltaTable.forName(spark, TARGET)
        dt.alias('t').merge(item_changes.alias('s'), 't.cdc_event_id = s.cdc_event_id') \
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        item_changes.write.format('delta').mode('overwrite').saveAsTable(TARGET)
    print(f'✅ silver.src_02_cdc.order_item_changes: {item_count} rows')

# ── PAYMENTS ──────────────────────────────────────────────────
payment_changes = changes.filter(F.col('cdc_table') == 'payments') \
    .withColumn('payment_id',      ext_long(F.col('after_raw'), F.lit('payment_id'))) \
    .withColumn('order_id',        ext_long(F.col('after_raw'), F.lit('order_id'))) \
    .withColumn('amount_pence',    ext_long(F.col('after_raw'), F.lit('amount_pence'))) \
    .withColumn('payment_status',  ext_str(F.col('after_raw'),  F.lit('payment_status'))) \
    .withColumn('payment_method',  ext_str(F.col('after_raw'),  F.lit('payment_method'))) \
    .withColumn('changed_at',      (F.col('cdc_ts_ms') / 1000).cast('timestamp')) \
    .select('cdc_event_id', 'cdc_op', 'changed_at', 'payment_id', 'order_id',
            'amount_pence', 'payment_status', 'payment_method', 'after_raw')

payment_count = payment_changes.count()
print(f'Payment changes: {payment_count}')

if payment_count > 0:
    TARGET = f'{SILVER_CATALOG}.src_02_cdc.payment_changes'
    if spark.catalog.tableExists(TARGET):
        dt = DeltaTable.forName(spark, TARGET)
        dt.alias('t').merge(payment_changes.alias('s'), 't.cdc_event_id = s.cdc_event_id') \
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        payment_changes.write.format('delta').mode('overwrite').saveAsTable(TARGET)
    print(f'✅ silver.src_02_cdc.payment_changes: {payment_count} rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
print('\n=== SILVER CDC TABLE COUNTS ===')
for table in ['order_changes', 'order_item_changes', 'payment_changes']:
    try:
        count = spark.sql(f'SELECT COUNT(*) as cnt FROM {SILVER_CATALOG}.src_02_cdc.{table}').collect()[0]['cnt']
        print(f'  silver.src_02_cdc.{table}: {count} rows')
    except:
        print(f'  silver.src_02_cdc.{table}: not created yet (no live CDC)')
